In [ ]:
import sqlite3

# Cria ou abre o arquivo do banco SQLite.
conexao = sqlite3.connect("loja.db")

# O cursor executa comandos SQL no banco.
cursor = conexao.cursor()

print("Banco conectado com sucesso!")



Banco conectado com sucesso!


In [ ]:
cursor.execute("PRAGMA foreign_keys = ON;")
print("Chaves estrangeiras ativadas!")


Chaves estrangeiras ativadas!


In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS cliente (
    id_cliente INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    telefone TEXT
)
""")

print("Tabela cliente criada!")


Tabela cliente criada!


In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS produto (
    id_produto INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    categoria TEXT,
    preco_atual REAL NOT NULL
)
""")

print("Tabela produto criada!")


Tabela produto criada!


In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS venda (
    id_venda INTEGER PRIMARY KEY AUTOINCREMENT,
    data TEXT NOT NULL,
    id_cliente INTEGER NOT NULL,

    FOREIGN KEY (id_cliente)
        REFERENCES cliente(id_cliente)
)
""")

print("Tabela venda criada!")


Tabela venda criada!


In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS item_venda (
    id_venda INTEGER NOT NULL,
    id_produto INTEGER NOT NULL,
    quantidade INTEGER NOT NULL CHECK (quantidade > 0),
    preco_praticado REAL NOT NULL CHECK (preco_praticado >= 0),
        PRIMARY KEY (id_venda, id_produto),

    FOREIGN KEY (id_venda)
        REFERENCES venda(id_venda),

    FOREIGN KEY (id_produto)
        REFERENCES produto(id_produto)
)
""")

conexao.commit()
print("Tabela item_venda criada e alterações confirmadas!")



Tabela item_venda criada e alterações confirmadas!


In [ ]:
cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""")

tabelas = cursor.fetchall()

for tabela in tabelas:
    print(tabela[0])


cliente
item_venda
produto
sqlite_sequence
venda


In [ ]:
cursor.execute("PRAGMA table_info(cliente);")

for coluna in cursor.fetchall():
    print(coluna)


(0, 'id_cliente', 'INTEGER', 0, None, 1)
(1, 'nome', 'TEXT', 1, None, 0)
(2, 'telefone', 'TEXT', 0, None, 0)


In [ ]:
clientes = [
    ("Ana", "9999-1111"),
    ("Carlos", "9999-2222"),
    ("Marina", "9999-3333")
]

cursor.executemany(
    "INSERT INTO cliente (nome, telefone) VALUES (?, ?)",
    clientes
)

conexao.commit()
print("Clientes inseridos!")


Clientes inseridos!


In [ ]:
produtos = [
    ("Mouse", "Informática", 80.00),
    ("Teclado", "Informática", 120.00),
    ("Monitor", "Informática", 950.00),
    ("Caderno", "Papelaria", 25.00)
]

cursor.executemany(
    """
    INSERT INTO produto (nome, categoria, preco_atual)
    VALUES (?, ?, ?)
    """,
    produtos
)

conexao.commit()
print("Produtos inseridos!")


Produtos inseridos!


In [ ]:
cursor.execute("SELECT * FROM cliente;")

for linha in cursor.fetchall():
    print(linha)


(1, 'Ana', '9999-1111')
(2, 'Carlos', '9999-2222')
(3, 'Marina', '9999-3333')


In [25]:
produtos = [
    ("Mouse", "Informática", 80.00),
    ("Teclado", "Informática", 120.00),
    ("Monitor", "Informática", 950.00),
    ("Caderno", "Papelaria", 25.00)
]

cursor.executemany(
    """
    INSERT INTO produto (nome, categoria, preco_atual)
    VALUES (?, ?, ?)
    """,
    produtos
)

conexao.commit()
print("Produtos inseridos!")


Produtos inseridos!


In [26]:
cursor.execute("SELECT * FROM cliente;")

for linha in cursor.fetchall():
    print(linha)


(1, 'Ana', '9999-1111')
(2, 'Carlos', '9999-2222')
(3, 'Marina', '9999-3333')


In [27]:
cursor.execute("SELECT * FROM produto ORDER BY nome;")

for linha in cursor.fetchall():
    print(linha)


(4, 'Caderno', 'Papelaria', 25.0)
(8, 'Caderno', 'Papelaria', 25.0)
(3, 'Monitor', 'Informática', 950.0)
(7, 'Monitor', 'Informática', 950.0)
(1, 'Mouse', 'Informática', 80.0)
(5, 'Mouse', 'Informática', 80.0)
(2, 'Teclado', 'Informática', 120.0)
(6, 'Teclado', 'Informática', 120.0)


In [28]:
cursor.execute(
    "INSERT INTO venda (data, id_cliente) VALUES (?, ?)",
    ("2026-10-05", 1)
)

id_venda_ana = cursor.lastrowid
print("ID da venda:", id_venda_ana)


ID da venda: 1


In [29]:
cursor.execute(
    """
    INSERT INTO item_venda
    (id_venda, id_produto, quantidade, preco_praticado)
    VALUES (?, ?, ?, ?)
    """,
    (id_venda_ana, 1, 2, 80.00)
)

cursor.execute(
    """
    INSERT INTO item_venda
    (id_venda, id_produto, quantidade, preco_praticado)
    VALUES (?, ?, ?, ?)
    """,
    (id_venda_ana, 2, 1, 120.00)
)

conexao.commit()
print("Itens da venda inseridos!")


Itens da venda inseridos!


In [30]:
cursor.execute("""
SELECT
    v.id_venda,
    v.data,
    c.nome AS cliente,
    p.nome AS produto,
    iv.quantidade,
    iv.preco_praticado,
    iv.quantidade * iv.preco_praticado AS subtotal
FROM item_venda iv
INNER JOIN venda v
    ON iv.id_venda = v.id_venda
INNER JOIN cliente c
    ON v.id_cliente = c.id_cliente
INNER JOIN produto p
    ON iv.id_produto = p.id_produto
ORDER BY v.id_venda, p.nome;
""")

for linha in cursor.fetchall():
    print(linha)


(1, '2026-10-05', 'Ana', 'Mouse', 2, 80.0, 160.0)
(1, '2026-10-05', 'Ana', 'Teclado', 1, 120.0, 120.0)


In [32]:
cursor.execute("""
SELECT
    v.id_venda,
    c.nome,
    v.data,
    SUM(iv.quantidade * iv.preco_praticado) AS total_venda
    FROM venda v
INNER JOIN cliente c
    ON v.id_cliente = c.id_cliente
INNER JOIN item_venda iv
    ON v.id_venda = iv.id_venda
GROUP BY v.id_venda, c.nome, v.data;
""")

for linha in cursor.fetchall():
    print(linha)



(1, 'Ana', '2026-10-05', 280.0)


In [33]:
cursor.execute("""
UPDATE produto
SET preco_atual = 95.00
WHERE id_produto = 1;
""")

conexao.commit()
print("Preço atual alterado!")


Preço atual alterado!


In [34]:
cursor.execute("SELECT nome, preco_atual FROM produto WHERE id_produto = 1;")
print("Produto:", cursor.fetchone())

cursor.execute("""
SELECT id_venda, id_produto, preco_praticado
FROM item_venda
WHERE id_produto = 1;
""")
print("Histórico da venda:", cursor.fetchall())


Produto: ('Mouse', 95.0)
Histórico da venda: [(1, 1, 80.0)]


In [35]:
try:
    cursor.execute(
        "INSERT INTO venda (data, id_cliente) VALUES (?, ?)",
        ("2026-10-06", 999)
    )
    conexao.commit()
except Exception as erro:
    print("Erro esperado:", erro)
    conexao.rollback()


Erro esperado: FOREIGN KEY constraint failed


In [36]:
try:
    cursor.execute(
        """
        INSERT INTO item_venda
        (id_venda, id_produto, quantidade, preco_praticado)
        VALUES (?, ?, ?, ?)
        """,
        (id_venda_ana, 3, 0, 950.00)
    )
    conexao.commit()
except Exception as erro:
    print("Erro esperado:", erro)
    conexao.rollback()


Erro esperado: CHECK constraint failed: quantidade > 0
